> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 9 · Notebook 08 — Robust parameters and a portfolio of pairs

**Sessions:** S8 (Robust optimization, portfolio of pairs & M5b release) · [Lesson plan](../../docs/lessons/PART_09_STATISTICAL_TRADING.md) · graded labs in [`labs/part09/`](../../labs/part09/)

**You will:**
1. See how much a pair's Sharpe depends on the entry and exit thresholds.
2. Replace one parameter choice with an ensemble of nearby ones.
3. Allocate a book of pairs with inverse volatility and per-name caps.
4. Look through the pairs to stock-level gross, net, beta and sector exposure.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters (true half-lives, true pairs, true hedge ratios, planted premia), so every estimator can be checked against the truth.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p9lib.py is in notebooks/part09/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p9lib as p

p.use_course_style()

## 1. One pair, 25 parameter choices

The pair and the formation are those of notebook 03. Here is its out-of-sample Sharpe for every (entry, exit) combination:

In [ ]:
pair = p.cointegrated_pair()
F = 750
eg = p.engle_granger(pair.y.iloc[:F], pair.x.iloc[:F])
W = round(3 * eg["half_life"])
z = p.rolling_zscore(pair.y - eg["beta"] * pair.x - eg["alpha"], W)
grid = {ex: {en: p.sharpe(p.pair_pnl(p.pairs_positions(z, en, ex), pair.y, pair.x, eg["beta"])[F:])
             for en in (1.5, 1.75, 2.0, 2.25, 2.5)} for ex in (0.0, 0.25, 0.5, 0.75, 1.0)}
pd.DataFrame(grid).rename_axis("entry ↓ · exit →").round(2)

Sharpe ranges from about −0.1 to 1.6 on the *same* pair and data. Picking the best cell is fitting noise (Part 8's overfitting lessons). A robust alternative: **average the positions** of several nearby parameter sets. The result is a fractional position in `[−1, 1]` that scales in and out.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def ensemble_positions(z, entries=(1.75, 2.0, 2.25), exits=(0.25, 0.5, 0.75), stop=4.0, max_hold=None):
    return np.mean([p.pairs_positions(z, e, x, stop, max_hold) for e in entries for x in exits], axis=0)

cases = [([0, 2.1, 1.0, 0.6, 0.3],), (z,), (z, (1.75, 2.0, 2.25), (0.25, 0.5, 0.75), 4.0, W)]
mine = [p.attempt(ensemble_positions, *c) for c in cases]
mine = p.check("ensemble_positions", mine, [p.ensemble_positions(*c) for c in cases])
print("toy:", np.round(mine[0], 3), "\nOOS Sharpe of the ensemble on this pair:",
      round(p.sharpe(p.pair_pnl(mine[1], pair.y, pair.x, eg["beta"])[F:]), 2))

On one pair, the ensemble lands inside the range of its nine members. Over several pairs, compare it with the members' worst, average and best, and with the textbook choice (2.0, 0.5), which you'd have picked in advance:

In [ ]:
rows = {}
for seed in range(10, 18):
    pr = p.cointegrated_pair(seed=seed)
    e2 = p.engle_granger(pr.y.iloc[:F], pr.x.iloc[:F])
    zz = p.rolling_zscore(pr.y - e2["beta"] * pr.x - e2["alpha"], round(3 * e2["half_life"]))
    member = [p.sharpe(p.pair_pnl(p.pairs_positions(zz, en, ex), pr.y, pr.x, e2["beta"])[F:])
              for en in (1.75, 2.0, 2.25) for ex in (0.25, 0.5, 0.75)]
    rows[f"pair {seed}"] = {"worst member": min(member), "mean member": np.mean(member), "best member": max(member),
                            "(2.0, 0.5)": member[4], "ensemble": p.sharpe(p.pair_pnl(p.ensemble_positions(zz), pr.y, pr.x, e2["beta"])[F:])}
table = pd.DataFrame(rows).T
display(table.round(2))
print(f"ensemble beats the mean member on {(table['ensemble'] > table['mean member']).sum()} of {len(table)} pairs")

The ensemble beats the average member on every pair and is never the worst. The textbook (2.0, 0.5) is sometimes better and sometimes far worse (pair 16). You don't know in advance which cell will be lucky; the ensemble doesn't need to.

## 2. A book of pairs

Five sectors of eight stocks with two true pairs per sector. Screen within sectors on the first 500 days, then trade every selected pair with the ensemble (look-back and time stop 3 half-lives) over the last 500.

In [ ]:
prices, sectors, truth = p.sector_universe(n_sectors=5, per_sector=8, pairs_per_sector=2, seed=3)
F = 500
screen = p.screen_pairs(prices.iloc[:F], sectors)
sel = screen[screen.selected]
pairs, hedges, rets, positions = {}, {}, {}, {}
for a, b in sel[["a", "b"]].to_numpy():
    key, y, x = f"{a}/{b}", prices[a], prices[b]
    e = p.engle_granger(y.iloc[:F], x.iloc[:F])
    hl = float(np.clip(e["half_life"], 2, 30))
    zz = p.rolling_zscore(y - e["beta"] * x - e["alpha"], max(10, round(3 * hl)))
    positions[key] = p.ensemble_positions(zz, max_hold=max(10, round(3 * hl)))
    rets[key] = p.pair_pnl(positions[key], y, x, e["beta"])
    pairs[key], hedges[key] = (a, b), e["beta"]
rets, positions, hedges = pd.DataFrame(rets, index=prices.index), pd.DataFrame(positions, index=prices.index), pd.Series(hedges)
print(f"{len(pairs)} pairs selected; not true pairs: {[k for k, ab in pairs.items() if ab not in truth]}; "
      f"true pairs missed: {[t for t in truth if t not in pairs.values()]}")
print("stocks in more than one pair:", pd.Series([n for ab in pairs.values() for n in ab]).value_counts().loc[lambda s: s > 1].to_dict())

One stock appears in two pairs, so its risk is counted twice. Weight the pairs by **inverse volatility** (formation returns), then cap every stock's **load** (the total weight of the pairs it appears in): repeat {scale each pair by `min(1, cap/load_a, cap/load_b)`; renormalize to 1} until no load exceeds the cap. If that never happens within `max_iter` rounds, the cap is infeasible.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pair_book_weights(pair_returns, pairs, name_cap=0.2, max_iter=1000):
    w = 1 / pair_returns.std(ddof=1)
    w = w / w.sum()                                       # inverse volatility, summing to 1
    for _ in range(max_iter):
        load = {}
        for k, (a, b) in pairs.items():                   # a stock's load: the weights of all pairs it is in
            load[a] = load.get(a, 0.0) + w[k]
            load[b] = load.get(b, 0.0) + w[k]
        if max(load.values()) <= name_cap * (1 + 1e-9):
            return w
        f = pd.Series({k: min(1.0, name_cap / load[a], name_cap / load[b]) for k, (a, b) in pairs.items()})
        w = w * f
        w = w / w.sum()
    raise ValueError(f"name cap {name_cap} is infeasible for these pairs")

formation_rets = rets.iloc[:F]
cases = [(formation_rets, pairs, 0.15), (formation_rets, pairs, 1.0)]
mine = [p.attempt(pair_book_weights, *c) for c in cases]
mine = p.check("pair_book_weights", mine, [p.pair_book_weights(*c) for c in cases])
w = mine[0]
display(pd.DataFrame({"inverse vol": mine[1], "with a 15% name cap": w}).round(3))
try:
    p.pair_book_weights(formation_rets, pairs, 0.1)
except ValueError as err:
    print("cap 0.10:", err)

In [ ]:
oos = rets.iloc[F:]
book = oos @ w
single = oos.apply(p.sharpe)
corr = oos.corr().to_numpy()[np.triu_indices(len(pairs), 1)].mean()
print(f"book Sharpe {p.sharpe(book):.2f}; single pairs: mean {single.mean():.2f} (from {single.min():.2f} to {single.max():.2f}); "
      f"average correlation between pairs {corr:+.3f}")
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(oos.cumsum(), lw=0.7, color="#8a8984"); ax.plot(book.cumsum(), lw=2.5, color="#2a78d6")
ax.set_title("single pairs (grey) and the book (blue), out of sample"); plt.show()

Nine weakly correlated bets with Sharpes around 1 add up to a book of about 3.6. Be suspicious of that number: here the pairs are independent **by construction** and none of them breaks. Real pairs share sectors and factors, and they fall together when crowded books unwind (Clinic W2 simulates one).

## 3. Look through to the stocks

Risk limits apply to stocks, not pairs. Pair `k` with position `s`, weight `w` and hedge `h` on names `(a, b)` puts `s·w/(1 + |h|)` on `a` and `−s·w·h/(1 + |h|)` on `b`; a stock collects weight from every pair it is in. Report the stock weights (zeros dropped), **gross** `Σ|w|`, **net** `Σw`, **net beta** `Σ w·β` and the net weight per sector. The stock betas are estimated on the formation period against an equal-weighted market.

In [ ]:
moves = prices.diff().iloc[1:F]
market = moves.mean(axis=1)
betas = moves.apply(lambda c: np.cov(c, market)[0, 1] / market.var())

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def book_exposures(pair_pos, pair_w, pairs, hedges, betas, sectors):
    w = {}
    for k, (a, b) in pairs.items():
        s, h = pair_pos[k] * pair_w[k], hedges[k]
        w[a] = w.get(a, 0.0) + s / (1 + abs(h))
        w[b] = w.get(b, 0.0) - s * h / (1 + abs(h))
    w = pd.Series(w).sort_index()
    w = w[w != 0]
    gross = float(w.abs().sum())
    return {"weights": w, "gross": gross, "net": float(w.sum()), "net_beta": float((w * betas.loc[w.index]).sum()),
            "sector_net": w.groupby(sectors.loc[w.index]).sum().sort_index()}

toy = (pd.Series({"P1": 1.0, "P2": -1.0}), pd.Series({"P1": 0.5, "P2": 0.5}), {"P1": ("A", "B"), "P2": ("A", "C")},
       pd.Series({"P1": 1.5, "P2": 0.5}), pd.Series({"A": 1.0, "B": 1.2, "C": 0.8}), pd.Series({"A": "tech", "B": "tech", "C": "fin"}))
today = (positions.iloc[-1], w, pairs, hedges, betas, sectors)
mine = [p.attempt(book_exposures, *toy), p.attempt(book_exposures, *today)]
mine = p.check("book_exposures", mine, [p.book_exposures(*toy), p.book_exposures(*today)])
ex = mine[1]
print(f"last day: {len(ex['weights'])} stocks, gross {ex['gross']:.3f}, net {ex['net']:+.4f}, net beta {ex['net_beta']:+.4f}")
ex["sector_net"].round(4).to_frame("net weight by sector")

In [ ]:
daily = pd.DataFrame([{k: v for k, v in p.book_exposures(positions.loc[d], w, pairs, hedges, betas, sectors).items()
                       if k in ("gross", "net", "net_beta")} for d in positions.index[F:]], index=positions.index[F:])
display(daily.describe().loc[["mean", "min", "max"]].round(3))
daily.plot(figsize=(10, 3), lw=1, title="the book's gross, net and net beta, day by day"); plt.show()

Each pair is close to dollar- and beta-neutral (most hedge ratios are near 1), so the book stays near zero net and zero net beta while its gross exposure moves with the number of open trades. That is what "market-neutral" should look like before you lean on it.

## Wrap-up

* Don't pick the best parameter cell; average nearby ones.
* Many small, weakly correlated pairs; weight by inverse volatility and cap every stock's load.
* Report and limit exposures at the stock level: gross, net, net beta, sector.
* A diversified book of pairs is not a hedge against a crowded unwind.
* Graded version: `labs/part09/week32_statarb` (`ensemble_positions`, `pair_book_weights`, `book_exposures`) and Clinic W2 (a 19-pair book, its risk report and a crowded-unwind scenario).